# Football Goal Forecasting

This Sporting Risk workshop exercise compares probability forecasts, then evaluates an independent Poisson model for home and away goals against a historical-average baseline.

## Comparing Probability Forecasts

In [12]:
import numpy as np

y  = np.array([1,0,1,1,0,0,1,0,0,1], dtype=float)
pA = np.array([0.60,0.55,0.62,0.58,0.51,0.49,0.61,0.52,0.47,0.59], dtype=float)
pB = np.array([0.75,0.40,0.70,0.65,0.35,0.30,0.80,0.25,0.20,0.70], dtype=float)

def logloss(y, p):
    # Natural log; clip to avoid log(0)
    eps = 1e-15
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

LLA = logloss(y, pA)
LLB = logloss(y, pB)

print("Forecast A LogLoss (mean):", LLA)
print("Forecast B LogLoss (mean):", LLB)

Forecast A LogLoss (mean): 0.6109567248922987
Forecast B LogLoss (mean): 0.3464067535299078


In [2]:

def brier(y, p):
    return np.mean((p - y)**2)

print("Forecast A Brier:", brier(y, pA))
print("Forecast B Brier:", brier(y, pB))

Forecast A Brier: 0.20950000000000002
Forecast B Brier: 0.08800000000000001


Under log loss, Forecaster B is superior because it achieves a lower value (about 0.346 vs 0.611 for A), meaning it assigned higher probability to the outcomes that actually occurred and is penalized more for being confidently wrong. Under the Brier score, Forecaster B is also superior because its mean squared error is smaller

## Forecasting Match Scores

In [3]:
import pandas as pd
import math

In [4]:
df=pd.read_excel("football_modelling.xlsx")

In [5]:

df.drop(columns=['Unnamed: 0'])
df.describe()

,Unnamed: 0,kickoff,home_team_id,away_team_id,home_team_score,away_team_score
count,281140.000000,281140,2.811400e+05,2.811400e+05,281140.000000,281140.000000
mean,283994.500000,2025-01-09 08:58:05.565697024,2.570935e+05,2.563299e+05,1.586918,1.307146
min,143425.000000,2024-01-01 06:00:00,1.000000e+00,1.000000e+00,0.000000,0.000000
25%,213709.750000,2024-07-14 09:00:00,6.070000e+03,6.015000e+03,1.000000,0.000000
50%,283994.500000,2025-01-19 11:00:00,7.514500e+04,7.447300e+04,1.000000,1.000000
75%,354279.250000,2025-07-30 19:00:00,3.965740e+05,3.959500e+05,2.000000,2.000000
max,424564.000000,2025-12-31 19:00:00,1.319310e+06,1.319310e+06,28.000000,30.000000
std,81158.271677,NaN,3.329228e+05,3.324750e+05,1.468722,1.349361


### Historical-Average Baseline

mean of home_team_score=1.586918
mean of away_team_score=1.307146

In [ ]:
# 0) cleaning
df = df.copy()
df["kickoff"] = pd.to_datetime(df["kickoff"], errors="coerce")
df = df.dropna(subset=["kickoff", "home_team_score", "away_team_score"])
df = df.sort_values("kickoff").reset_index(drop=True)

# 1) time-based 80/20 split
split_idx = int(0.8 * len(df))
train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

# 2) mean=λ
lambda_home = train["home_team_score"].mean()
lambda_away = train["away_team_score"].mean()

print("lambda_home (train mean):", lambda_home)
print("lambda_away (train mean):", lambda_away)

# 3) Poisson Negative Log-Likelihood (NLL) on test

eps = 1e-12
lh = max(lambda_home, eps)
la = max(lambda_away, eps)

yH = test["home_team_score"].to_numpy()
yA = test["away_team_score"].to_numpy()

nll_home = np.mean(lh - yH*np.log(lh) + np.vectorize(math.lgamma)(yH + 1))
nll_away = np.mean(la - yA*np.log(la) + np.vectorize(math.lgamma)(yA + 1))


nll_total = nll_home + nll_away

print("Poisson NLL (home) on test:", nll_home)
print("Poisson NLL (away) on test:", nll_away)
print("Poisson NLL (home+away) on test:", nll_total)

lambda_home (train mean): 1.5803291954186527
lambda_away (train mean): 1.3050926584619762
Poisson NLL (home) on test: 1.680519934477067
Poisson NLL (away) on test: 1.5527804604679145
Poisson NLL (home+away) on test: 3.2333003949449814


### Team-Level Poisson Model


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance

# 1) long-format
train_long = pd.concat([
    pd.DataFrame({
        "goals": train["home_team_score"].astype(int),
        "is_home": 1,
        "attack_team": train["home_team_id"].astype(str),
        "defence_team": train["away_team_id"].astype(str),
    }),
    pd.DataFrame({
        "goals": train["away_team_score"].astype(int),
        "is_home": 0,
        "attack_team": train["away_team_id"].astype(str),
        "defence_team": train["home_team_id"].astype(str),
    })
], ignore_index=True)

test_home = pd.DataFrame({
    "is_home": 1,
    "attack_team": test["home_team_id"].astype(str),
    "defence_team": test["away_team_id"].astype(str),
})
test_away = pd.DataFrame({
    "is_home": 0,
    "attack_team": test["away_team_id"].astype(str),
    "defence_team": test["home_team_id"].astype(str),
})

X_train = train_long[["is_home","attack_team","defence_team"]]
y_train = train_long["goals"].to_numpy()

# 2)Poisson regression
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", ["is_home"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), ["attack_team","defence_team"]),
    ],
    sparse_threshold=1.0,  
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("glm", PoissonRegressor(alpha=1e-6, max_iter=1000))  
])

model.fit(X_train, y_train)

# 3)  test - λ_hat_home, λ_hat_away
lambda_hat_home = model.predict(test_home)
lambda_hat_away = model.predict(test_away)

test_out = test.copy()
test_out["lambda_hat_home"] = lambda_hat_home
test_out["lambda_hat_away"] = lambda_hat_away

print(test_out[["kickoff","home_team_id","away_team_id","lambda_hat_home","lambda_hat_away"]].head())

# 4)  mean Poisson deviance evaluation
home_dev = mean_poisson_deviance(test["home_team_score"], lambda_hat_home)
away_dev = mean_poisson_deviance(test["away_team_score"], lambda_hat_away)
print("Test mean Poisson deviance (home):", home_dev)
print("Test mean Poisson deviance (away):", away_dev)
print("Total:", home_dev + away_dev)

                   kickoff  home_team_id  away_team_id  lambda_hat_home  \
224912 2025-08-30 15:30:00       1214735        621106         1.081194   
224913 2025-08-30 15:30:00        760704       1176555         1.189065   
224914 2025-08-30 15:30:00         33821         33818         0.855097   
224915 2025-08-30 15:30:00        600490          2081         1.236990   
224916 2025-08-30 15:30:00         37864         36195         1.532257   

        lambda_hat_away  
224912         0.723330  
224913         0.808429  
224914         1.731489  
224915         2.513625  
224916         1.140143  
Test mean Poisson deviance (home): 1.3529718338168895
Test mean Poisson deviance (away): 1.3368526874382234
Total: 2.6898245212551126


/opt/homebrew/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


#### Model Specification

### 1) Model definition
For match $i$ with home team $h(i)$ and away team $a(i)$, let $HG_i$ be home goals and $AG_i$ be away goals.

Assume conditional independence given the intensities:
$HG_i \mid \lambda_{H,i} \sim \text{Poisson}(\lambda_{H,i})$, $AG_i \mid \lambda_{A,i} \sim \text{Poisson}(\lambda_{A,i})$, and $HG_i \perp AG_i \mid (\lambda_{H,i},\lambda_{A,i})$.

Use a log-link and model goal intensities via team-specific attack/defence effects plus home advantage:
$\log \lambda_{H,i}=\mu+\gamma+\alpha_{h(i)}+\beta_{a(i)}$, and $\log \lambda_{A,i}=\mu+\alpha_{a(i)}+\beta_{h(i)}$.

Here $\mu$ is a global intercept, $\gamma$ is the home-advantage parameter, $\alpha_t$ is the attack strength of team $t$, and $\beta_t$ is the defence effect of team $t$. Parameters are fitted on the training set by maximum likelihood under the Poisson GLM.

### 2) Identifiability treatment
To ensure identifiability, we use reference-category (treatment) coding for both $\{\alpha_t\}$ and $\{\beta_t\}$: one reference team $t_0$ has $\alpha_{t_0}=0$ and $\beta_{t_0}=0$, and all other team effects are interpreted relative to $t_0$.

### 3) Producing $\hat\lambda_H,\hat\lambda_A$ for test matches
For each test match $i$, predicted intensities are computed as
$\hat\lambda_{H,i}=\exp(\hat\mu+\hat\gamma+\hat\alpha_{h(i)}+\hat\beta_{a(i)})$ and
$\hat\lambda_{A,i}=\exp(\hat\mu+\hat\alpha_{a(i)}+\hat\beta_{h(i)})$.

### Held-Out Evaluation

In [ ]:

def poisson_nll(y, lam):
    """
    Mean Poisson negative log-likelihood:
    NLL = mean( lam - y*log(lam) + log(y!) )
    with log(y!) = lgamma(y+1)
    """
    y = np.asarray(y, dtype=float)
    lam = np.asarray(lam, dtype=float)
    eps = 1e-12
    lam = np.clip(lam, eps, None)
    lgamma = np.vectorize(math.lgamma)
    return np.mean(lam - y*np.log(lam) + lgamma(y + 1))

# NLL on test 
yH_test = test_out["home_team_score"].to_numpy()
yA_test = test_out["away_team_score"].to_numpy()

lamH_model = test_out["lambda_hat_home"].to_numpy()
lamA_model = test_out["lambda_hat_away"].to_numpy()

nllH_model = poisson_nll(yH_test, lamH_model)
nllA_model = poisson_nll(yA_test, lamA_model)

#Baseline NLL on test 

lambda_home_base = train["home_team_score"].mean()
lambda_away_base = train["away_team_score"].mean()

lamH_base = np.full_like(yH_test, lambda_home_base, dtype=float)
lamA_base = np.full_like(yA_test, lambda_away_base, dtype=float)

nllH_base = poisson_nll(yH_test, lamH_base)
nllA_base = poisson_nll(yA_test, lamA_base)

# Improvement (baseline - model)
imp_H = nllH_base - nllH_model
imp_A = nllA_base - nllA_model

imp_H_pct = imp_H / nllH_base * 100
imp_A_pct = imp_A / nllA_base * 100

print("Test NLL (Home) - model   :", nllH_model)
print("Test NLL (Away) - model   :", nllA_model)
print("Test NLL (Home) - baseline:", nllH_base)
print("Test NLL (Away) - baseline:", nllA_base)
print("Improvement Home (abs, %)  :", imp_H, imp_H_pct)
print("Improvement Away (abs, %)  :", imp_A, imp_A_pct)

Test NLL (Home) - model   : 1.639131476784415
Test NLL (Away) - model   : 1.5081615922149858
Test NLL (Home) - baseline: 1.6804630172149408
Test NLL (Away) - baseline: 1.552797073105547
Improvement Home (abs, %)  : 0.04133154043052589 2.459532879160015
Improvement Away (abs, %)  : 0.04463548089056113 2.874521189126891


## C4：Model Critique

Structural weakness： 
Independent Poisson assumes $HG \perp AG \mid (\lambda_H,\lambda_A)$ and $\text{Var}(Y)=\mathbb{E}[Y]=\lambda$, so it cannot capture dependence between teams’ goal counts (e.g. game state effects) and often underestimates variability / low-score structure.

Concrete improvement
Implement a Dixon–Coles adjustment (add a parameter $\rho$ to modify the joint probabilities of 0–0, 0–1, 1–0, 1–1), which introduces dependence in low scores and typically improves fit for football scorelines.